In [1]:
!python helpers/generate_poisson_ar1.py

Simulated DataFrame:
              timestamp  y1  y2
0  2024-01-01T00:00:00Z   4   3
1  2024-01-01T01:00:00Z   6   6
2  2024-01-01T02:00:00Z   2   6
3  2024-01-01T03:00:00Z   1   3
4  2024-01-01T04:00:00Z   3   1
Simulated data saved to 'simulated_poisson_ar1.parquet'


In [6]:
# from raw data to features
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .master("local[*]") \
    .getOrCreate()


In [7]:
# read from parquet file
df_sim = spark.read.parquet("simulated_poisson_ar1.parquet")
df_sim.show(5)

+--------------------+---+---+
|           timestamp| y1| y2|
+--------------------+---+---+
|2024-01-01T00:00:00Z|  4|  3|
|2024-01-01T01:00:00Z|  6|  6|
|2024-01-01T02:00:00Z|  2|  6|
|2024-01-01T03:00:00Z|  1|  3|
|2024-01-01T04:00:00Z|  3|  1|
+--------------------+---+---+
only showing top 5 rows



In [10]:
from pyspark.sql import functions as sf
from pyspark.sql.window import Window

# compute lagged values
df_sim = df_sim.withColumn("y1_lag1", sf.lag("y1", 1).over(Window.orderBy("timestamp")))
df_sim = df_sim.withColumn("y2_lag1", sf.lag("y2", 1).over(Window.orderBy("timestamp")))
df_sim.show(5)

+--------------------+---+---+-------+-------+
|           timestamp| y1| y2|y1_lag1|y2_lag1|
+--------------------+---+---+-------+-------+
|2024-01-01T00:00:00Z|  4|  3|   NULL|   NULL|
|2024-01-01T01:00:00Z|  6|  6|      4|      3|
|2024-01-01T02:00:00Z|  2|  6|      6|      6|
|2024-01-01T03:00:00Z|  1|  3|      2|      6|
|2024-01-01T04:00:00Z|  3|  1|      1|      3|
+--------------------+---+---+-------+-------+
only showing top 5 rows



In [11]:
# filter NULL values and save to features.parquet
df_sim.filter(sf.col("y1_lag1").isNotNull() & sf.col("y2_lag1").isNotNull()) \
    .select("timestamp", "y1", "y2", "y1_lag1", "y2_lag1") \
    .write.mode("overwrite").parquet("features.parquet")
